# Lab 02 — Surrogate keys na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: **gerar** uma surrogate key para uma dimensão e fazer o **surrogate key lookup** ao carregar o fato.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
con = duckdb.connect()
# Staging vindo da origem: produtos com a CHAVE NATURAL (codigo)
con.execute('CREATE TABLE stg_produto(codigo VARCHAR, categoria VARCHAR)')
con.executemany('INSERT INTO stg_produto VALUES (?,?)', [
    ('P-100','eletronicos'),('P-050','livros'),('P-200','casa'),('P-010','livros')])
con.execute('SELECT * FROM stg_produto').df()

## 1. Gerar a dimensão com surrogate key
`ROW_NUMBER()` cria um inteiro sequencial, sem significado — a **surrogate key**. A dimensão guarda a surrogate **e** a chave natural (`codigo`).

In [ ]:
con.execute('''
    CREATE TABLE dim_produto AS
    SELECT ROW_NUMBER() OVER (ORDER BY codigo) AS sk_produto, codigo, categoria
    FROM stg_produto
''')
con.execute('SELECT * FROM dim_produto ORDER BY sk_produto').df()

## 2. Surrogate key lookup ao carregar o fato
As vendas chegam com a **chave natural** (`codigo_produto`). Fazemos JOIN com a dimensão pela chave natural para trocar pelo `sk_produto` — o fato **nunca** guarda 'P-100'.

In [ ]:
con.execute('CREATE TABLE stg_venda(venda_id INT, codigo_produto VARCHAR, valor DOUBLE)')
con.executemany('INSERT INTO stg_venda VALUES (?,?,?)', [
    (1,'P-100',1200.0),(2,'P-010',30.0),(3,'P-200',80.0),(4,'P-100',1500.0)])
con.execute('''
    SELECT v.venda_id, d.sk_produto, v.valor
    FROM stg_venda v
    JOIN dim_produto d ON v.codigo_produto = d.codigo
    ORDER BY v.venda_id
''').df()

## 3. Sua vez (mini-desafio)
Monte o **fato final** com colunas `(venda_id, sk_produto, valor)` fazendo o lookup, ordenado por `venda_id`. Devolva `.fetchall()` e verifique.

In [ ]:
resposta = con.execute('''
    SELECT v.venda_id, d.sk_produto, v.valor
    FROM stg_venda v
    JOIN dim_produto d ON v.codigo_produto = d.codigo
    ORDER BY v.venda_id
''').fetchall()
resposta

In [ ]:
def verificar(rows):
    esperado = [(1,3,1200.0),(2,1,30.0),(3,4,80.0),(4,3,1500.0)]
    try:
        assert rows == esperado, 'Confira o JOIN pela chave natural e a ordem.'
        print('✅ Correto! Você trocou a chave natural pela surrogate key (lookup).')
    except AssertionError as e:
        print('❌', e)

verificar(resposta)